# 05 - Explainability, Risk Engine, ECL, Portfolio, Fairness, Monitoring

**Purpose:** SHAP, PD risk bands, illustrative ECL, portfolio analytics, subgroup fairness audit, drift monitoring.

**Inputs:** creditiq_bundle.joblib, holdout_features.parquet, holdout_scored.parquet

**Outputs / persisted artifacts:** shap_*.csv/png, portfolio_summary.json, ecl_summary.json, portfolio_*.csv, fairness_*.csv, drift_report.csv

**Dependencies:** Notebook 04

**Execution order:** 5 of 6

**Validation checks:** Bands cover [0,1]; ECL = PD x LGD x EAD; observed rate and predicted PD reported separately

_All logic lives in `src/creditiq`; cells are safe to re-run (completed stages are skipped, artifacts live on Google Drive)._

In [ ]:
# --- Colab setup (safe to re-run) ---
import os, sys, subprocess
from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
ROOT = Path("/content/drive/MyDrive/CreditIQ") if Path("/content/drive/MyDrive").exists() else Path.cwd().parent
os.environ["CREDITIQ_ROOT"] = str(ROOT); sys.path.insert(0, str(ROOT / "src"))
if not (ROOT / "src").exists():
    raise SystemExit(f"Project not found at {ROOT}. Unzip CreditIQ into MyDrive/CreditIQ first.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=False)
from creditiq.config import get_paths, load_config, read_state
paths = get_paths(); cfg = load_config(paths)
print("Project root:", ROOT); print("Last completed stage:", read_state(paths).get("last_completed"))

In [ ]:
from creditiq import pipeline
pipeline.run("analytics")

In [ ]:
from creditiq.risk_engine import RiskEngine
import pandas as pd
eng = RiskEngine(paths=paths); hf = pd.read_parquet(paths.processed / "holdout_features.parquet")
res = eng.assess(hf.head(3))
for r in res: print({k: r[k] for k in ("applicant_id", "pd", "risk_band", "recommendation")}, "|", r["explanation"])

In [ ]:
import json
display(pd.read_csv(paths.metrics / "shap_global_importance.csv").head(20))
from IPython.display import Image; display(Image(str(paths.figures / "shap_summary.png")))
print(json.dumps(json.load(open(paths.metrics / "portfolio_summary.json")), indent=2)); print(json.dumps(json.load(open(paths.metrics / "ecl_summary.json")), indent=2))
display(pd.read_csv(paths.reports / "fairness_disparity.csv")); display(pd.read_csv(paths.metrics / "drift_report.csv").head(20))